# UDepth — FLSea

UDepth-RMI pré-treinado, sem treinamento adicional. Este notebook leve chama
`evaluate_flsea.py` e usa os mesmos frames, máscaras e métricas do grupo.

A avaliação registrada contém 119 imagens válidas dos 120 pares em seis sequências
do subconjunto `arxiv-2507.02148`. O CSV e a documentação ficam versionados;
os mapas completos estão no [Drive de resultados](https://drive.google.com/drive/folders/10tms9tiqb5yA4rWIOqZ9tEEsMNGv8O9p).

Na revisão da PR, use a branch `feat/udepth-flsea`; após o merge, pode usar `main`.

## 1. Repositório e ambiente

O Colab já oferece PyTorch. A instalação abaixo usa as dependências declaradas
pelo projeto. Em uma sessão nova, o código é obtido da branch configurada.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/AndreFGard/Yolo-Depth-Underwater.git'
REVISAO = 'feat/udepth-flsea'
REPO = Path.cwd().resolve()
if not (REPO/'src/yolo_depth_underwater/udepth_model.py').is_file():
    REPO = (Path('/content') if Path('/content').exists() else Path.cwd())/'Yolo-Depth-Underwater'
    if not REPO.exists():
        subprocess.run(['git','clone','--branch',REVISAO,REPO_URL,str(REPO)],check=True)
    if not (REPO/'src/yolo_depth_underwater/udepth_model.py').is_file():
        raise RuntimeError('Abra um checkout da branch com a integração UDepth.')
print('Checkout:',subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)+'[flsea,udepth]'],check=True)
sys.path.insert(0,str(REPO/'src'))


## 2. Dados e destino

Informe `FLSEA_LOCAL` para usar o FLSea-VI v5 já disponível. Deixe vazio para usar
o download parcial do projeto, cujo handle aponta para a versão disponível no Kaggle.
Para reproduzir exatamente os resultados registrados, use a versão 5.
O download público funcionou sem credenciais na execução original; autentique-se
fora do código se o serviço solicitar. Não é necessário baixar o ZIP completo.

Com `MONTAR_DRIVE=True`, confira a conta Google na autorização. Os resultados
serão gravados no Drive dessa conta; o notebook não escolhe a conta automaticamente.

In [ ]:
MONTAR_DRIVE = True
DISPOSITIVO = 'auto'
FLSEA_LOCAL = ''
SUBCONJUNTO = 'arxiv-2507.02148'

if MONTAR_DRIVE and Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/projeto_profundidade')
else:
    BASE = REPO/'results'
SAIDA = BASE/'UDepth_FLSea_reproducao'
SAIDA.mkdir(parents=True,exist_ok=True)
os.environ['UDEPTH_ASSETS'] = str(BASE/'udepth_assets')
os.environ['UDEPTH_CACHE'] = str(SAIDA/'cache')
os.environ['KAGGLEHUB_CACHE'] = str(BASE/'cache_flsea')
print('Destino:',SAIDA)


## 3. Inferência e avaliação

A rede recebe RMI 480×640 e mantém os pesos congelados. O avaliador recebe a
representação inversa relativa, maior para mais perto. Os mapas finais NPY contêm
`1/max(inversa,1e-6)`, maior para mais longe, sem normalização min–max nem uso de GT.

Os PNGs em `alinhadas_ssi/` usam o GT apenas para avaliação e não representam
profundidade métrica independente. `ssi` e `median` são alinhamentos por imagem;
`global-ssi` ajusta escala e deslocamento com a outra metade dos frames.

In [ ]:
command = [sys.executable,'-u',str(REPO/'scripts/evaluate_flsea.py'),
           '--model','udepth','--subset',SUBCONJUNTO,
           '--out',str(SAIDA/'udepth_evaluation_flsea.csv'),
           '--save-predictions',str(SAIDA/'predicoes_udepth')]
if FLSEA_LOCAL:
    command += ['--local-root',FLSEA_LOCAL]
if DISPOSITIVO != 'auto':
    command += ['--device',DISPOSITIVO]
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO/'src') + os.pathsep + env.get('PYTHONPATH','')
subprocess.run(command,cwd=REPO,env=env,check=True)


## 4. Resultados e arquivos

A tabela abaixo mostra a média por imagem do CSV desta execução. Os mapas relativos
ficam em `predicoes_udepth/relativas/<sequência>/<frame_id>.npy`, acompanhados de
`indice_relativas.csv` e dos hashes SHA-256.

In [ ]:
import pandas as pd
from IPython.display import display

csv_path = SAIDA/'udepth_evaluation_flsea.csv'
tabela = pd.read_csv(csv_path,dtype={'frame_id':str})
display(tabela.groupby('alignment',sort=False)[['AbsRel','RMSE','d1','d2','d3','SILog']].mean())
print('Imagens válidas:',len(tabela[['sequence','frame_id']].drop_duplicates()))
print('CSV:',csv_path)
print('Mapas relativos:',SAIDA/'predicoes_udepth/relativas')
print('Índice:',SAIDA/'predicoes_udepth/indice_relativas.csv')


## Conclusão do experimento registrado

O UDepth-RMI foi avaliado sem ajuste de pesos nos mesmos **119 frames válidos** do
grupo. O CSV versionado foi produzido em CPU usando o avaliador do commit
`da0166ce07c2ac05a813ad122010d505d236acf4` e o FLSea-VI v5.

Com **SSI no espaço inverso**, a média por imagem foi **AbsRel 0,1291**, **RMSE
alinhado 0,9278**, **δ1 84,66%**, **δ2 96,17%**, **δ3 98,70%** e **SILog 17,5406**.
δ1 mede a proporção de pixels válidos com razão previsão/referência menor que
1,25, e não uma porcentagem de imagens corretas.

O SSI usa a referência da própria imagem, portanto mede a estrutura após
alinhamento. Com `global-ssi`, δ1 foi **33,89%** e AbsRel **0,3496**, inferiores ao
controle constante nessas duas métricas (**39,97%** e **0,3230**). Isso evidencia
limitações de calibração entre imagens. Os mapas NPY permanecem em profundidade
relativa; esses resultados não demonstram profundidade em metros para imagens novas.

A tabela da seção anterior pertence à execução atual e pode variar com o ambiente.
O piloto anterior de 150 imagens não faz parte desta comparação.